(Rachel Young's HW1 to get the pickled files)

# Homework 1 | Handling Data

A wise figure once said to me: to do anything useful in life, you need to be able to handle 8 gigabytes of option data. The dataset you all have is SPX option data from 1996 to 2023. FUN FACT: I recieved this data as a part of a Summer Reserach Experience I did; the project was paid for, but I found it quite interesting that this SPX daily data costed 3,000 dollars...

In order to really get crack-a-lackin with this dataset, we need a systematic and efficient way to access the contents. Right now the data is stored in a bunch of .zip files on your computers. Your job will be to get the data from a .zip format into a .pkl format (pickle)

Why do this? Well, .zip formats need to be unzipped in order to be accessed. So that's why.

As great thinkers, we will NOT be unzipping the 3,000+ contents of the folder I gave you one by one... Instead, you will make Python do this for you!

## Tasks (not necessarily in order)

### 1. Unzip each of the files and place the result in a seperate folder

### 2. For each unzipped file, convert it to a pkl format

### 3. Rename each file --> SPX_OPTIONS_MMDDYYYY

### 4. Open up a file and play around with the data :)


In [2]:
import os
import zipfile
import pandas as pd
from datetime import datetime

In [2]:
from pathlib import Path
import zipfile
import io
import re
import pandas as pd


def _parse_txt_to_df(bytes_data: bytes) -> pd.DataFrame:
    """Try a few parsers to turn a TXT file into a DataFrame."""
    buf = io.BytesIO(bytes_data)

    # 1) Let pandas infer the delimiter (engine='python' allows sep=None)
    for kwargs in (dict(sep=None, engine="python"),
                   dict(delim_whitespace=True, engine="python")):
        buf.seek(0)
        try:
            df = pd.read_csv(buf, **kwargs)
            # If it parsed to at least 1 row or multiple columns, accept it
            if len(df) > 0 or df.shape[1] > 1:
                return df
        except Exception:
            pass

    # 2) Try fixed-width as a last structured attempt
    buf.seek(0)
    try:
        df = pd.read_fwf(buf)
        if len(df) > 0:
            return df
    except Exception:
        pass

    # 3) Absolute fallback: store raw lines in a single column
    text = bytes_data.decode("utf-8", errors="ignore")
    return pd.DataFrame({"raw_line": text.splitlines()})


def convert_zip_folder(folder_path: str):
    base = Path(folder_path).expanduser().resolve()
    out_dir = base / "SPX pkl files"
    out_dir.mkdir(exist_ok=True)

    zips = sorted(base.glob("*.zip"))
    if not zips:
        print(f"No .zip files found in: {base}")
        return

    for zpath in zips:
        # Expect something like IVYOPPRCD_199601.zip → yyyymm = 199601
        m = re.search(r"(\d{6})", zpath.stem)
        if not m:
            print(f"Skipping {zpath.name} (no yyyymm in name)")
            continue

        yyyymm = m.group(1)
        yyyy, mm = yyyymm[:4], yyyymm[4:]  # "1996", "01"
        out_name = f"SPX_OPTIONS_{mm}{yyyy}.pkl"
        out_path = out_dir / out_name

        if out_path.exists():
            print(f"[skip] {out_name} already exists")
            continue

        # Read the TXT file from inside the zip, in memory
        with zipfile.ZipFile(zpath) as zf:
            members = [n for n in zf.namelist() if n.lower().endswith(".txt")]
            if not members:
                print(f"!! No .txt found in {zpath.name}")
                continue
            if len(members) > 1:
                print(f":: Multiple .txt files in {zpath.name}; using {members[0]}")
            txt_bytes = zf.read(members[0])

        # Convert to DataFrame, then pickle
        df = _parse_txt_to_df(txt_bytes)
        df.to_pickle(out_path)
        print(f"Saved {out_name}  ({df.shape[0]} rows × {df.shape[1]} cols)")

    print(f"\nAll done. Pickles are in: {out_dir}")


if __name__ == "__main__":
    # >>> EDIT THIS LINE to the folder that contains your IVYOPPRCD_*.zip files
    convert_zip_folder(r"C:\TAMID\OPPRCD_SPX")




[skip] SPX_OPTIONS_011996.pkl already exists
[skip] SPX_OPTIONS_021996.pkl already exists
[skip] SPX_OPTIONS_031996.pkl already exists
[skip] SPX_OPTIONS_041996.pkl already exists
[skip] SPX_OPTIONS_051996.pkl already exists
[skip] SPX_OPTIONS_061996.pkl already exists
[skip] SPX_OPTIONS_071996.pkl already exists
[skip] SPX_OPTIONS_081996.pkl already exists
[skip] SPX_OPTIONS_091996.pkl already exists
[skip] SPX_OPTIONS_101996.pkl already exists
[skip] SPX_OPTIONS_111996.pkl already exists
[skip] SPX_OPTIONS_121996.pkl already exists
[skip] SPX_OPTIONS_011997.pkl already exists
[skip] SPX_OPTIONS_021997.pkl already exists
[skip] SPX_OPTIONS_031997.pkl already exists
[skip] SPX_OPTIONS_041997.pkl already exists
[skip] SPX_OPTIONS_051997.pkl already exists
[skip] SPX_OPTIONS_061997.pkl already exists
[skip] SPX_OPTIONS_071997.pkl already exists
[skip] SPX_OPTIONS_081997.pkl already exists
[skip] SPX_OPTIONS_091997.pkl already exists
[skip] SPX_OPTIONS_101997.pkl already exists
[skip] SPX